In [1]:
!pip install pandas ipython-sql
import pandas as pd
import sqlite3
from google.colab import files
%load_ext sql
print("✅ Setup complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.6 MB/s eta 0:00:00
✅ Setup complete!


In [ ]:
print("📁 Upload retail_sales.csv")
files.upload()

📁 Upload retail_sales.csv


Saving retail_sales.csv to retail_sales (2).csv


{'retail_sales (2).csv': b'\xef\xbb\xbftransactions_id,sale_date,sale_time,customer_id,gender,age,category,quantiy,price_per_unit,cogs,total_sale\r\n180,2022-11-05,10:47:00,117,Male,41,Clothing,3,300,129,900\r\n522,2022-07-09,11:00:00,52,Male,46,Beauty,3,500,145,1500\r\n559,2022-12-12,10:48:00,5,Female,40,Clothing,4,300,84,1200\r\n1180,2022-01-06,08:53:00,85,Male,41,Clothing,3,300,129,900\r\n1522,2022-11-14,08:35:00,48,Male,46,Beauty,3,500,235,1500\r\n1559,2022-08-20,07:40:00,49,Female,40,Clothing,4,300,144,1200\r\n163,2022-10-31,09:38:00,144,Female,64,Clothing,3,50,23,150\r\n303,2022-04-22,11:09:00,54,Male,19,Electronics,3,30,14.7,90\r\n421,2022-04-08,08:43:00,66,Female,37,Clothing,3,500,235,1500\r\n979,2022-05-18,10:18:00,6,Female,19,Beauty,1,25,10.5,25\r\n1163,2022-05-04,10:52:00,120,Female,64,Clothing,3,50,27,150\r\n1303,2022-03-19,08:59:00,58,Male,19,Electronics,3,30,15,90\r\n1421,2022-01-17,07:07:00,59,Female,37,Clothing,3,500,185,1500\r\n1979,2022-08-17,11:34:00,102,Female,19,Be

In [ ]:
# Cell 3: Create Database + Load Data
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect('retail.db')
cursor = conn.cursor()

# Drop table if exists
cursor.execute('DROP TABLE IF EXISTS retail_sales')

# Create table with exact schema from GitHub project
cursor.execute('''
CREATE TABLE retail_sales (
    transaction_id INTEGER PRIMARY KEY,
    sale_date TEXT,
    sale_time TEXT,
    customer_id INTEGER,
    gender TEXT,
    age INTEGER,
    category TEXT,
    quantity INTEGER,
    price_per_unit REAL,
    cogs REAL,
    total_sale REAL
)
''')

# Load CSV automatically
df = pd.read_csv('retail_sales.csv')
df.to_sql('retail_sales', conn, if_exists='replace', index=False)
print(f"✅ Loaded {len(df)} rows!")

# Commit changes
conn.commit()

✅ Loaded 2000 rows!


In [ ]:
# Test: Count records
test_query = "SELECT COUNT(*) as total_records FROM retail_sales"
result = pd.read_sql_query(test_query, conn)
print(f"Total records in database: {result['total_records'].iloc[0]}")
result

Total records in database: 2000


,total_records
0,2000


In [ ]:
# Cell 1: Check your CSV columns
import pandas as pd

df = pd.read_csv('retail_sales.csv')
print("Column names in your CSV:")
print(df.columns.tolist())
print("\nFirst few rows:")
df.head()

Column names in your CSV:
['transactions_id', 'sale_date', 'sale_time', 'customer_id', 'gender', 'age', 'category', 'quantiy', 'price_per_unit', 'cogs', 'total_sale']

First few rows:


,transactions_id,sale_date,sale_time,customer_id,gender,age,category,quantiy,price_per_unit,cogs,total_sale
0,180,2022-11-05,10:47:00,117,Male,41.0,Clothing,3.0,300.0,129.0,900.0
1,522,2022-07-09,11:00:00,52,Male,46.0,Beauty,3.0,500.0,145.0,1500.0
2,559,2022-12-12,10:48:00,5,Female,40.0,Clothing,4.0,300.0,84.0,1200.0
3,1180,2022-01-06,08:53:00,85,Male,41.0,Clothing,3.0,300.0,129.0,900.0
4,1522,2022-11-14,08:35:00,48,Male,46.0,Beauty,3.0,500.0,235.0,1500.0


In [ ]:
# Cell 2: Example - View first 10 rows
query = "SELECT * FROM retail_sales LIMIT 10"
pd.read_sql_query(query, conn)

,transactions_id,sale_date,sale_time,customer_id,gender,age,category,quantiy,price_per_unit,cogs,total_sale
0,180,2022-11-05,10:47:00,117,Male,41.0,Clothing,3.0,300.0,129.0,900.0
1,522,2022-07-09,11:00:00,52,Male,46.0,Beauty,3.0,500.0,145.0,1500.0
2,559,2022-12-12,10:48:00,5,Female,40.0,Clothing,4.0,300.0,84.0,1200.0
3,1180,2022-01-06,08:53:00,85,Male,41.0,Clothing,3.0,300.0,129.0,900.0
4,1522,2022-11-14,08:35:00,48,Male,46.0,Beauty,3.0,500.0,235.0,1500.0
5,1559,2022-08-20,07:40:00,49,Female,40.0,Clothing,4.0,300.0,144.0,1200.0
6,163,2022-10-31,09:38:00,144,Female,64.0,Clothing,3.0,50.0,23.0,150.0
7,303,2022-04-22,11:09:00,54,Male,19.0,Electronics,3.0,30.0,14.7,90.0
8,421,2022-04-08,08:43:00,66,Female,37.0,Clothing,3.0,500.0,235.0,1500.0
9,979,2022-05-18,10:18:00,6,Female,19.0,Beauty,1.0,25.0,10.5,25.0


In [ ]:
# Cell 3: Count by category
query = """
SELECT
    category,
    COUNT(*) as total_transactions,
    SUM(total_sale) as total_sales
FROM retail_sales
GROUP BY category
"""
pd.read_sql_query(query, conn)

,category,total_transactions,total_sales
0,Beauty,614,286840.0
1,Clothing,702,311070.0
2,Electronics,684,313810.0


In [ ]:
%%sql
-- Q0: Check NULLs (should be 0 after cleaning)
SELECT COUNT(*) as null_records
FROM retail_sales
WHERE sale_date IS NULL OR customer_id IS NULL OR category IS NULL OR total_sale IS NULL;

-- Clean NULLs (if any found)

DELETE FROM retail_sales

WHERE sale_date IS NULL OR customer_id IS NULL OR category IS NULL OR total_sale IS NULL;

 * sqlite:///retail.db
Done.
3 rows affected.


[]

In [ ]:
query = """
SELECT COUNT(*) as total_transactions,
       COUNT(DISTINCT customer_id) as unique_customers,
       COUNT(DISTINCT category) as categories
FROM retail_sales
"""
result = pd.read_sql_query(query, conn)
print(result)

   total_transactions  unique_customers  categories
0                1997               155           3


In [ ]:
query = """
SELECT DISTINCT category
FROM retail_sales
ORDER BY category
"""
result = pd.read_sql_query(query, conn)
print(result)

      category
0       Beauty
1     Clothing
2  Electronics


In [ ]:
query = """
SELECT category,
       SUM(total_sale) as net_sales,
       COUNT(*) as total_orders,
       ROUND(AVG(total_sale),2) as avg_order_value
FROM retail_sales
GROUP BY category
ORDER BY net_sales DESC
"""
result = pd.read_sql_query(query, conn)
print(result)

      category  net_sales  total_orders  avg_order_value
0  Electronics   313810.0           684           458.79
1     Clothing   311070.0           701           443.75
2       Beauty   286840.0           612           468.69


In [ ]:
query = """
SELECT ROUND(AVG(age), 2) as avg_age_beauty,
       COUNT(*) as beauty_orders
FROM retail_sales
WHERE category = 'Beauty'
"""
result = pd.read_sql_query(query, conn)
print(result)

   avg_age_beauty  beauty_orders
0           40.42            612


In [ ]:
query = """
SELECT transactions_id, customer_id, total_sale, category
FROM retail_sales
WHERE total_sale > 1000
ORDER BY total_sale DESC
"""
result = pd.read_sql_query(query, conn)
print(result)

     transactions_id  customer_id  total_sale     category
0                 15           75      2000.0  Electronics
1                743           55      2000.0       Beauty
2               1015           94      2000.0  Electronics
3               1743           47      2000.0       Beauty
4                742           37      2000.0  Electronics
..               ...          ...         ...          ...
301             1099           56      1200.0  Electronics
302              757           82      1200.0  Electronics
303             1757           50      1200.0  Electronics
304              908           64      1200.0       Beauty
305             1908           93      1200.0       Beauty

[306 rows x 4 columns]


In [ ]:
query = """
SELECT category, gender,
       COUNT(*) as transactions,
       SUM(total_sale) as revenue
FROM retail_sales
GROUP BY category, gender
ORDER BY category, revenue DESC
"""
result = pd.read_sql_query(query, conn)
print(result)

      category  gender  transactions   revenue
0       Beauty  Female           330  149470.0
1       Beauty    Male           282  137370.0
2     Clothing  Female           347  162460.0
3     Clothing    Male           354  148610.0
4  Electronics    Male           344  160340.0
5  Electronics  Female           340  153470.0


In [ ]:
query = """
SELECT customer_id,
       COUNT(*) as orders,
       SUM(total_sale) as total_spent
FROM retail_sales
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 5
"""
result = pd.read_sql_query(query, conn)
print(result)

   customer_id  orders  total_spent
0            3      76      38440.0
1            1      76      30750.0
2            5      63      30405.0
3            2      69      25295.0
4            4      73      23580.0


In [ ]:
query = """
SELECT
    CASE
        WHEN CAST(SUBSTR(sale_time,1,2) AS INTEGER) < 12 THEN 'Morning'
        WHEN CAST(SUBSTR(sale_time,1,2) AS INTEGER) <= 17 THEN 'Afternoon'
        ELSE 'Evening'
    END as shift,
    COUNT(*) as orders,
    SUM(total_sale) as revenue
FROM retail_sales
GROUP BY 1
ORDER BY revenue DESC
"""
result = pd.read_sql_query(query, conn)
print(result)

       shift  orders   revenue
0    Evening    1062  475940.0
1    Morning     558  259900.0
2  Afternoon     377  175880.0
